<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Solution: Isolation Forest

**[Isolation Forest](https://learning.nextia-ai.com/courses/anomaly-detection/m04/isolation-forest/)** · Anomaly Detection: Find What Is Unusual · Module 4, lesson 3 of 5 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** find unusual points by cutting the data at random and counting how fast each point is alone, run Isolation Forest on every payment, and choose its threshold from the alert budget instead of the contamination setting.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (about 7 MB). No account. The notebook takes about a minute to run. |
| **You should know** | kNN distance and scaling, from [Far from the neighbours: kNN distance](https://learning.nextia-ai.com/courses/anomaly-detection/m04/far-from-the-neighbours-knn-distance/). |
| **You do not need** | A local project. The setup below downloads everything. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m04/isolation-forest/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M04-L03-isolation-forest/isolation-forest-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib']:
    print(p, version(p))

## Setup: get the course data

Run the next cell. It downloads the data that this lesson uses (`payments.csv.gz`, `sealed-key.txt`) into a folder `anomaly-course/data`, and checks each file's checksum. It also downloads `compare.py` and `features.py`, the course's shared code. It needs an internet connection and no account. It takes about 10 seconds.

You should see `Ready:` and the files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course: no real people, cards or payments. [The dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md) describes every table.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/"
FILES = {
    "payments.csv.gz": "c4e18c1bbaf422cfc99821f2a1b65cf0e8274f835c40ca757a37a531eeccd542",
    "sealed-key.txt": "144cfd16737bdfcc4e992ad8ede0b8856265a2193c89fa7266fe0b27cb62ff98",
}
CODE_FILES = ["compare.py", "features.py"]
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A guessing game: how many cuts to get a point alone?

Think of the game "Guess who?". You ask yes/no questions to find one person. A person with an unusual trait (the only one with a red hat) is found in one question. A person who looks like many others needs many questions.

**Isolation Forest** plays this game with random questions. It picks a random column and a random value between the smallest and the largest, and cuts the points in two. It keeps cutting the side with the target until the target is alone. A strange point is alone after a few cuts; a normal point needs many.

Run the next cell. It cuts the seven points of the kNN lesson until each one is alone, with one random tree (seed 0).

> **Check.** You should see G = 1 and A = 6.

In [ ]:
points = np.array([[1.0, 1.0], [1.2, 0.9], [0.9, 1.1], [1.1, 1.2], [0.8, 0.8], [1.0, 1.3], [3.0, 3.0]])
names = ["A", "B", "C", "D", "E", "F", "G"]

def cuts_to_isolate(target, seed):
    """Cut the points at random (a random column, a random value) until `target` is alone. Count the cuts."""
    rng = np.random.default_rng(seed)
    alive = np.arange(len(points))
    cuts = 0
    while len(alive) > 1:
        col = int(rng.integers(2))                          # 0 = x, 1 = y
        low, high = points[alive, col].min(), points[alive, col].max()
        if low == high:
            continue
        value = rng.uniform(low, high)                      # a random cut between the smallest and the largest
        same_side = (points[alive, col] < value) == (points[target, col] < value)
        alive = alive[same_side]                            # keep only the side of the target
        cuts += 1
    return cuts

print("one random tree (seed 0):", {n: cuts_to_isolate(i, seed=0) for i, n in enumerate(names)})

One tree is luck. A **forest** is many random trees, and the score is the average number of cuts. Run the next cell: 1,000 trees, one per seed.

> **Check.** You should see G = 1.21 on average, and 3.02 to 4.56 for A to F.

In [ ]:
average = {n: round(float(np.mean([cuts_to_isolate(i, seed) for seed in range(1000)])), 2) for i, n in enumerate(names)}
print("average over 1,000 trees:", average)

## 2. Larkfield: every payment, every reference row

Run the next cell. It loads the payments, builds the behaviour features and scales them on the reference months. This time, the detector learns from **all** 173,271 reference payments, not a sample: Isolation Forest does not compare points with each other, so it stays fast.

In [ ]:
import time
import compare, features
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
payments = pd.read_csv("data/payments.csv.gz", parse_dates=["ts"])
reference, evaluation = compare.split(payments)
behaviour = features.behaviour(payments)
scaler = StandardScaler().fit(behaviour.loc[reference.index])
X_ref, X_ev = scaler.transform(behaviour.loc[reference.index]), scaler.transform(behaviour.loc[evaluation.index])
print(X_ref.shape, X_ev.shape)

Run the next cell. scikit-learn's `IsolationForest` grows 200 trees, each on a random sample of 256 reference payments. `score_samples` gives minus the score, so the code turns it around: about 0.5 is average, close to 1 means "alone after very few cuts".

> **Check.** You should see a precision at the budget of 0.500, and a median score of 0.378 for honest payments against 0.742 for planted ones.

In [ ]:
forest = IsolationForest(n_estimators=200, random_state=0, n_jobs=-1).fit(X_ref)   # all 173,271 reference payments
scores = -forest.score_samples(X_ev)          # about 0.5 = average; close to 1 = isolated in very few cuts
table = compare.Table()
table.add(compare.score_payments(evaluation, scores, "Isolation Forest", "behaviour features, 200 trees",
                                 explain_minutes=3))
table.show()
planted = evaluation.payment_id.isin(compare.planted_payments().payment_id).to_numpy()
print("median score, honest:", np.median(scores[~planted]).round(3), "| planted:", np.median(scores[planted]).round(3))

## 3. The contamination setting: a guess, not a fact

`IsolationForest` can also say "anomaly" or "normal" by itself, with `predict`. For that, it needs a threshold, and it takes it from **contamination**: the share of anomalies that you expect. The default, `"auto"`, uses a fixed rule from the original paper. It does not know your data.

> **Predict.** About 1.7% of the evaluation payments are planted. What share will `contamination="auto"` flag?

> **Check.** You should see `"auto"` flag 20.7% (386.3 a day, precision 0.079), and 0.01 flag 25.7 a day.

In [ ]:
days = evaluation.ts.dt.floor("D").nunique()
print("planted payments a day:", round(planted.sum() / days, 1), "| Sofia's budget: 50 a day")
for c in ["auto", 0.01]:
    flagged = IsolationForest(n_estimators=200, contamination=c, random_state=0, n_jobs=-1).fit(X_ref).predict(X_ev) == -1
    print(f"contamination={c!r}: flags {flagged.mean():.1%} = {flagged.sum() / days:.1f} a day, "
          f"precision {planted[flagged].mean():.3f}")

`"auto"` flags far more payments than Sofia's team can review. A guess of 0.01 is closer, but you only know that because you have the answer key. A real team does not know the share of anomalies. So the course never uses `predict`: it ranks by score and takes the top 50 a day, the **alert budget**.

## 4. Speed on many rows

Run the next cell. It fits and scores all 343,074 payments at once. (Your time will differ: it depends on your computer and on what else runs.)

In [ ]:
X_all = scaler.transform(behaviour)
start = time.time()
IsolationForest(n_estimators=200, random_state=0, n_jobs=-1).fit(X_all).score_samples(X_all)
print(f"{len(X_all):,} payments, fit and score: {time.time() - start:.1f} seconds")

## 5. Check yourself

**Your turn 1.** On average over the 1,000 trees, how many cuts does G need? Write it as `g_average` (2 decimals).

In [ ]:
g_average = 1.21

In [ ]:
expect('g_average', g_average, 1.21, tolerance=0.01)

**Your turn 2.** With `contamination="auto"`, how many payments a day does Isolation Forest flag in the evaluation months? Write it as `auto_per_day` (1 decimal). Compare it with the budget of 50.

In [ ]:
auto_per_day = 386.3

In [ ]:
expect('auto_per_day', auto_per_day, 386.3, tolerance=0.002)

**Your turn 3.** Do you need 200 trees? Fit a forest with only 10 trees (`random_state=0`) on `X_ref`, score `X_ev`, and put the precision at the budget in `precision_10`.

In [ ]:
small = IsolationForest(n_estimators=10, random_state=0, n_jobs=-1).fit(X_ref)
precision_10 = compare.score_payments(evaluation, -small.score_samples(X_ev), "Isolation Forest",
                                      "behaviour features, 10 trees")["precision at budget"]
print(precision_10)

In [ ]:
expect('precision_10', precision_10, 0.49, tolerance=0.001)

In the solution: 0.490 with 10 trees, against 0.500 with 200. Few trees give a noisier score; more trees cost a little more time.

## Recap

- Isolation Forest cuts the data at random. A strange point is alone after few cuts (G: 1.21 on average); a normal point needs more (3.02 to 4.56).
- It needs no distances between points, so it learns from every reference payment and stays fast.
- With the behaviour features: 0.500 at 50 alerts a day.
- `contamination="auto"` flags 386 payments a day: choose the threshold from the budget, not from a guess.

The next lesson draws a boundary around normal data, and runs four detectors with one interface.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [One-Class SVM and the PyOD toolbox](https://learning.nextia-ai.com/courses/anomaly-detection/m04/one-class-svm-and-the-pyod-toolbox/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m04/isolation-forest/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.